In [467]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
TOKEN = "7227666723:AAEsumQ2gWyr582xK3kGDwMFej0IvX1wD0s"
chat_id = "220684438"
# import threading
mt5.initialize()


True

In [468]:
def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

In [469]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("GBPJPY", 1.0, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

13.47

In [470]:
def calculate_heikin_ashi(df):
    ha_close = (df['open'] + df['high'] + df['low'] + df['close']) / 4
    ha_open = (ha_close.shift(1) + ha_close.shift(1)) / 2
    ha_high = df[['high', 'open', 'close']].max(axis=1)
    ha_low = df[['low', 'open', 'close']].min(axis=1)

    return pd.DataFrame({'ha_open': ha_open, 'ha_high': ha_high, 'ha_low': ha_low, 'ha_close': ha_close, 'time':df.time})

In [471]:
import numpy as np
import pandas as pd
import pandas_ta as pdt

def supertrend(factor, atr_length, high, low, close):
    atr = pdt.atr(high, low, close, atr_length)
    basic_upper_band = (high + low) / 2 + factor * atr
    basic_lower_band = (high + low) / 2 - factor * atr
    bullish_signal = close > basic_upper_band
    bearish_signal = close < basic_lower_band
    bullish_supertrend = np.full_like(close, np.nan)
    bearish_supertrend = np.full_like(close, np.nan)

    for i in range(1, len(close)):
        if bullish_signal[i] or (bullish_supertrend[i-1] and close[i-1] > basic_upper_band[i-1]):
            bullish_supertrend[i] = max(basic_upper_band[i], bullish_supertrend[i-1])
        else:
            bullish_supertrend[i] = basic_upper_band[i]

        if bearish_signal[i] or (bearish_supertrend[i-1] and close[i-1] < basic_lower_band[i-1]):
            bearish_supertrend[i] = min(basic_lower_band[i], bearish_supertrend[i-1])
        else:
            bearish_supertrend[i] = basic_lower_band[i]

    direction = np.where(close > bullish_supertrend, 1, np.where(close < bearish_supertrend, -1, 0))
    supertrend = np.where(direction == 1, bullish_supertrend, bearish_supertrend)
    
    return supertrend, direction

# Example usage:
# Assuming df is your DataFrame containing OHLC data
# Replace this with your actual DataFrame
# Example:
# df = pd.DataFrame({'open': [...], 'high': [...], 'low': [...], 'close': [...]})

# Convert input parameters from Pine Script to Python
# factor = 3.0
# atr_length = 10




In [472]:
def get_values(symbol, size, smaa=150, t='M5'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
#     rates_frame = rates_frame[rates_frame['sma'].notna()]
    # Calculate Supertrend
    factor = 3.0
    atr_length = 10
    supertrend_values, direction = supertrend(factor, atr_length, rates_frame['high'], rates_frame['low'], rates_frame['close'])
    rates_frame['spvalues'] = supertrend_values
    rates_frame['direction'] = direction
    
    # Print or access the supertrend_values and direction arrays
    print("Supertrend values:", supertrend_values)
    print("Direction:", direction)
    return rates_frame

In [473]:
def get_values(symbol, size, smaa=50, t='M30'):
    d = {'M1':mt5.TIMEFRAME_M1, 'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
    rates_frame['ema1'] =rates_frame['close'].ewm(span=9, adjust=False).mean()
    rates_frame['ema2'] =rates_frame['close'].ewm(span=15, adjust=False).mean()
    rates_frame['ema3'] =rates_frame['close'].ewm(span=100, adjust=False).mean()

#     rates_frame['ema'] =ema(rates_frame['close'], 9)
#     rates_frame['ema'] =ema(rates_frame['close'], 9)
    rates_frame['rsi1'] = get_rsi(rates_frame['close'], 7)
    rates_frame['rsi2'] = get_rsi(rates_frame['close'], 14)
    
    
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
    rates_frame = rates_frame[rates_frame['sma'].notna()]
#         print(rates_frame.head())
    # Calculate Supertren
    return rates_frame

In [474]:
def ema(s, n):
    ema = []
    zero = [0]*(20000-19801)
    j = 1

    #get n sma first and calculate the next n period ema
    sma = sum(s[:n]) / n
    multiplier = 2 / float(1 + n)
    ema.append(sma)

    #EMA(current) = ( (Price(current) - EMA(prev) ) x Multiplier) + EMA(prev)
    ema.append(( (s[n] - sma) * multiplier) + sma)

    #now calculate the rest of the values
    for i in s[n+1:]:
        tmp = ( (i - ema[j]) * multiplier) + ema[j]
        j = j + 1
        ema.append(tmp)
    am = zero + ema
    print(len(am))
    return am

In [475]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [495]:
NEED TO REVISIT THIS AGAIN, RSI1 AND 14 LESS THAN 50 AT THE SAME TIME
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 20000, 25, 'M1')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)


for i in range(1, len(a)):
    if check==0:
        if a.iloc[i-3].rsi1 >50.0 and a.iloc[i-3].rsi2 > 50.0 and a.iloc[i-2].rsi1 < 50.0 and a.iloc[i-1].rsi1 < a.iloc[i-2].rsi1 and a.iloc[i-1].rsi2 < 50 and a.iloc[i-1].rsi2 < a.iloc[i-2].rsi2:
            print("=="*20)
            print(f"{a.iloc[i].name}")
            c = 0
           
            buy_price = a.iloc[i].close
            check=1
            
#         if a.iloc[i-1].close >= a.iloc[i-1].sma and direction(a,i-1)==1:
#             print(f"{a.iloc[i].name}")
#             buy_price = a.iloc[i].open
#             check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].close
        lot = 0.1
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10))
#         ppb = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
        print(f"PP {pp}--{ a.iloc[i].rsi2}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp > 0.50:
            c = 1
        if pp <= -3.0:
            print(f"{pp}--{ a.iloc[i].rsi2}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            if c !=1:
                profit.append(-3)
            else:
                profit.append(0.30)
            check = 0
        if (a.iloc[i].ema + 24.0) < sell_price:
#             ppp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10))
#             print(f"PPP {ppp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            if pp<0.0:
                if c!=1:
                    profit.append(pp)
                else:
                    profit.append(0.30)
            else:
                profit.append(pp)
            check =0
#         elif pp<-15:
#             profit.append(-15)
#             check =0
#         else:
#             profit.append(pp)
#         check=0

    if check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-5:
            profit.append(-5)
        else:
            profit.append(pp)
        check=0

SyntaxError: invalid syntax (39566123.py, line 1)

In [360]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 40000, 5, 'M5')
timezone =pytz.timezone('Etc/GMT-2')
lot = 0.1

# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0

def loss():
    return price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL)
for i in range(10, len(a)-5):
        
    if check==0:        
        if a.iloc[i].close < a.iloc[i].ema1 and a.iloc[i].close < a.iloc[i].ema2 and direction(a, i) == 0 and \
        a.iloc[i-1].close < a.iloc[i-1].ema1 and a.iloc[i-1].close < a.iloc[i-1].ema2 and direction(a, i-1) == 0 and \
        a.iloc[i-2].close < a.iloc[i-2].ema1 and a.iloc[i-2].close < a.iloc[i-2].ema2 and direction(a, i-2) == 0 and \
        price_action(symbol, lot, a.iloc[i].close, a.iloc[i].ema1, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10)) > -10.0 and \
        price_action(symbol, lot, a.iloc[i].close, a.iloc[i].ema2, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10)) > -10.0:
            print("=="*20)
            print(f"{a.iloc[i].name} !! {a.iloc[i].ema1} !! {a.iloc[i].ema2} !! {a.iloc[i].close}")
            c = 0
            buy_price = a.iloc[i].close
            pp_old = 0.0
            check=1
            checks = 0
            up = 0
            hpp = 0.0
            
#         if "00:05" in str(a.iloc[i+1].name.time())[:6] and a.iloc[i].close > a.iloc[i].open:
#             print("=="*20)
#             print(f"{a.iloc[i].name} !! {a.iloc[i].rsi1} !! {a.iloc[i].rsi2}")
#             c = 0
#             buy_price = a.iloc[i].close
#             check=2
#             continue
    elif check==1:
        sell_price = a.iloc[i].close
        lot = 0.1
        if c!=0:
            pp_old = pp
            up=1
#             print(f"up===>{up}")
        c+=1
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10))
        ppopen = price_action(symbol, 2, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
        print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price}--{a.iloc[i].name}")
        if hpp < pp_old:
            hpp = pp_old
        if pp > 3.0:
            up=1
            
        if pp>=10.0:
#             print(f"PP {pp}-- { a.iloc[i+1].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
#             profit.append(pp)
            checks=1
        if a.iloc[i].rsi1 < 10 and a.iloc[i].rsi1 > a.iloc[i-1].rsi1 and (a.iloc[i-1].rsi1 - a.iloc[i].rsi1) >=1:
            print(f"rsipp {pp}-- { a.iloc[i+1].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
#             if pp > 0.0:
#                 profit.append(pp)
#             elif pp<0.0 and up==1:
#                 profit.append(-1)
#             else:
#                 profit.append(pp)
            profit.append(pp)
            check=0
            continue
        if sell_price > a.iloc[i-1].ema1 + 20 or sell_price > a.iloc[i-1].ema2+ 20:
            print(f"ema_cross {pp}-- { a.iloc[i+1].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
            if pp < -10:
                if up !=1:
                    profit.append(-10)
                else:
                    profit.append(-1)
            else:
                if pp > 0.0:
                    profit.append(pp)
                elif pp<0.0 and up==1:
                    profit.append(-1)
                else:
                    profit.append(pp)
            check=0
            continue
        if pp < hpp/1.75 and checks!=0:
            print(f"HPP_old {hpp/1.75}-- { a.iloc[i+1].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
            profit.append(hpp/2)
            check =0
            continue
            


2024-04-29 21:10:00 !! 62945.5554734497 !! 62946.3506136776 !! 62876.33
PP -9.4-- 50.073167214249146--- 51.70090624462483--62876.33--2024-04-29 21:15:00
PP -5.91-- 45.56500124902545--- 49.64477753550248--62876.33--2024-04-29 21:20:00
PP -6.75-- 46.900634983737--- 50.155995585159566--62876.33--2024-04-29 21:25:00
PP 6.989999999999999-- 31.8956337866353--- 42.51667876769834--62876.33--2024-04-29 21:30:00
PP 7.04-- 31.85063814392504--- 42.490300008284436--62876.33--2024-04-29 21:35:00
PP 10.21-- 28.94803943473768--- 40.828330066880376--62876.33--2024-04-29 21:40:00
PP 10.86-- 28.329105455319805--- 40.47795561113769--62876.33--2024-04-29 21:45:00
PP 9.55-- 31.76216199658407--- 41.56712762916245--62876.33--2024-04-29 21:50:00
PP 2.5199999999999996-- 47.48086257704774--- 47.14540615717117--62876.33--2024-04-29 21:55:00
HPP_old 6.2057142857142855-- 25.022432309638944--- 47.14540615717117--62876.33 !! 62829.15--2024-04-29 21:55:00
2024-05-02 10:50:00 !! 57648.125669970985 !! 57634.4508086023 !

2024-05-12 01:20:00 !! 60920.793357124836 !! 60927.2091742663 !! 60906.97
PP -4.62-- 51.880042389733454--- 47.6163950862704--60906.97--2024-05-12 01:25:00
PP -8.32-- 61.060510170623395--- 52.08958850921223--60906.97--2024-05-12 01:30:00
ema_cross -8.32-- 53.89762258341484--- 52.08958850921223--60906.97 !! 60968.15--2024-05-12 01:30:00
2024-05-12 04:30:00 !! 60925.63307944503 !! 60922.80314599352 !! 60891.5
PP -1.0300000000000002-- 37.85265283666995--- 44.85962811046623--60891.5--2024-05-12 04:35:00
PP -2.0100000000000002-- 41.50693697874893--- 46.30462745216009--60891.5--2024-05-12 04:40:00
PP 0.6199999999999997-- 35.03745740679618--- 43.03558466871963--60891.5--2024-05-12 04:45:00
PP 2.1399999999999997-- 31.69994353262898--- 41.22103473454441--60891.5--2024-05-12 04:50:00
PP 3.2299999999999995-- 29.349652767252408--- 39.91503306173121--60891.5--2024-05-12 04:55:00
PP 1.4099999999999997-- 38.24874848710505--- 43.14690299104898--60891.5--2024-05-12 05:00:00
PP 6.499999999999999-- 27.133

2024-05-20 06:25:00 !! 66640.69664556463 !! 66632.5901271184 !! 66593.86
PP -6.6000000000000005-- 49.97403990284685--- 54.63399699006353--66593.86--2024-05-20 06:30:00
PP -12.43-- 61.08415040781928--- 59.084497432320035--66593.86--2024-05-20 06:35:00
ema_cross -12.43-- 64.8966134262212--- 59.084497432320035--66593.86 !! 66696.14--2024-05-20 06:35:00
2024-05-20 11:40:00 !! 66893.10678706829 !! 66883.6288881257 !! 66857.15
PP -2.24-- 45.10792972561541--- 48.532239729450644--66857.15--2024-05-20 11:45:00
PP 4.7-- 35.41157244123512--- 44.07917353530498--66857.15--2024-05-20 11:50:00
PP 5.96-- 33.865928787467624--- 43.30050902329735--66857.15--2024-05-20 11:55:00
PP -1.5100000000000002-- 49.172253353037895--- 49.03426282889747--66857.15--2024-05-20 12:00:00
PP -11.11-- 62.270607555873745--- 55.293984185610775--66857.15--2024-05-20 12:05:00
ema_cross -11.11-- 62.35664093184559--- 55.293984185610775--66857.15 !! 66946.25--2024-05-20 12:05:00
2024-05-20 14:40:00 !! 67157.40061439318 !! 67145.6

2024-05-26 14:30:00 !! 69133.68772951767 !! 69135.4293253529 !! 69083.09
PP -1.0500000000000003-- 27.533757200712813--- 40.701535783679404--69083.09--2024-05-26 14:35:00
PP -2.3000000000000003-- 34.25511456809227--- 42.94671146497445--69083.09--2024-05-26 14:40:00
PP -0.6300000000000003-- 29.914776011227005--- 40.72046075827453--69083.09--2024-05-26 14:45:00
PP -2.3800000000000003-- 39.333313782841905--- 44.00351347227601--69083.09--2024-05-26 14:50:00
PP -5.18-- 51.43537113734349--- 48.85215814597703--69083.09--2024-05-26 14:55:00
PP -3.89-- 46.471384250605446--- 46.84714738451262--69083.09--2024-05-26 15:00:00
PP -5.18-- 51.92264193937077--- 49.11216803571409--69083.09--2024-05-26 15:05:00
PP 2.8999999999999995-- 29.75805143285477--- 38.13968129743384--69083.09--2024-05-26 15:10:00
PP 7.4799999999999995-- 23.210169026176587--- 33.5650065053766--69083.09--2024-05-26 15:15:00
PP 8.8-- 21.61606928214985--- 32.36405258183706--69083.09--2024-05-26 15:20:00
PP 5.0-- 36.359990072858956--- 3

2024-05-28 01:50:00 !! 69544.42840210508 !! 69555.35794911542 !! 69478.57
PP -4.61-- 41.873880502157334--- 45.06988585300419--69478.57--2024-05-28 01:55:00
PP 2.17-- 33.30736478825973--- 40.96039311279639--69478.57--2024-05-28 02:00:00
PP 1.9699999999999998-- 33.76669052930629--- 41.128362096232685--69478.57--2024-05-28 02:05:00
PP 3.13-- 32.2525611016942--- 40.40506744481045--69478.57--2024-05-28 02:10:00
PP 11.129999999999999-- 23.64996855915568--- 35.703460422962934--69478.57--2024-05-28 02:15:00
PP 14.129999999999999-- 21.18088977380056--- 34.10257180346068--69478.57--2024-05-28 02:20:00
PP 14.54-- 20.827196630711356--- 33.874509681336164--69478.57--2024-05-28 02:25:00
PP 15.280000000000001-- 20.133736083415897--- 33.448620042742036--69478.57--2024-05-28 02:30:00
PP 19.59-- 16.417702019064507--- 31.002723025326134--69478.57--2024-05-28 02:35:00
PP 15.190000000000001-- 31.480337835129248--- 36.13698168565195--69478.57--2024-05-28 02:40:00
PP 14.129999999999999-- 34.79833556622769---

...

2024-08-28 00:05:00 !! 61923.66398186832 !! 61949.19755281111 !! 61872.86
PP 28.36-- 15.367680468117271--- 29.57510670344925--61872.86--2024-08-28 00:10:00
PP 47.349999999999994-- 10.902838462198886--- 24.448449894661564--61872.86--2024-08-28 00:15:00
PP 33.68-- 28.37399469629395--- 33.39591164989001--61872.86--2024-08-28 00:20:00
PP 45.419999999999995-- 23.7163442354353--- 30.10036477786855--61872.86--2024-08-28 00:25:00
PP 62.22-- 18.613414535429484--- 26.12580613749175--61872.86--2024-08-28 00:30:00
PP 87.77-- 13.470349933776632--- 21.480004405610387--61872.86--2024-08-28 00:35:00
PP 77.64-- 23.277698700286734--- 27.021866159633262--61872.86--2024-08-28 00:40:00
PP 161.25-- 11.131299281786426--- 16.60602971299656--61872.86--2024-08-28 00:45:00
PP 163.68-- 10.93730957948263--- 16.407592903885288--61872.86--2024-08-28 00:50:00
PP 143.25-- 23.91445370349831--- 24.552452341963487--61872.86--2024-08-28 00:55:00
PP 154.29000000000002-- 21.902088857351984--- 23.234712069164047--61872.86--2

PP 0.2999999999999998-- 42.684147798065574--- 44.28279342179423--58909.67--2024-08-31 22:35:00
PP 4.34-- 35.08060667650996--- 40.50295070516911--58909.67--2024-08-31 22:40:00
PP 4.9799999999999995-- 33.96861335780481--- 39.92486117585241--58909.67--2024-08-31 22:45:00
PP 3.88-- 37.92214971123188--- 41.47406541262183--58909.67--2024-08-31 22:50:00
PP 2.7299999999999995-- 42.17150623819432--- 43.13533547567304--58909.67--2024-08-31 22:55:00
PP 7.260000000000001-- 32.09289190237587--- 38.50652332539987--58909.67--2024-08-31 23:00:00
PP 9.89-- 27.641528174843245--- 36.0971894970883--58909.67--2024-08-31 23:05:00
PP 4.6-- 45.448188834996266--- 43.74366152969875--58909.67--2024-08-31 23:10:00
PP -1.8900000000000001-- 59.66877242090672--- 51.43016074469263--58909.67--2024-08-31 23:15:00
ema_cross -1.8900000000000001-- 48.074418274910705--- 51.43016074469263--58909.67 !! 58906.62--2024-08-31 23:15:00
2024-09-01 01:50:00 !! 59025.100897044475 !! 59021.90278837985 !! 59006.48
PP -3.0700000000000

2024-09-07 17:40:00 !! 54548.972585723044 !! 54548.8451086161 !! 54481.79
PP -4.71-- 40.54648543034084--- 47.85623168482218--54481.79--2024-09-07 17:45:00
PP -17.95-- 64.47328730958446--- 59.80136303265212--54481.79--2024-09-07 17:50:00
ema_cross -17.95-- 61.172349481015225--- 59.80136303265212--54481.79 !! 54639.31--2024-09-07 17:50:00
2024-09-08 06:20:00 !! 54322.179051368345 !! 54318.76133189932 !! 54290.19
PP -3.02-- 41.93496919627736--- 51.54760539267595--54290.19--2024-09-08 06:25:00
PP 3.6899999999999995-- 27.18375123846033--- 43.18854116653699--54290.19--2024-09-08 06:30:00
PP -0.8200000000000003-- 42.93587920740053--- 49.1604517043913--54290.19--2024-09-08 06:35:00
PP 0.08000000000000007-- 40.86406438982604--- 48.067341880862934--54290.19--2024-09-08 06:40:00
PP -2.89-- 50.07190975926136--- 51.84489555637804--54290.19--2024-09-08 06:45:00
PP -7.73-- 61.49310080102602--- 57.30556133676427--54290.19--2024-09-08 06:50:00
ema_cross -7.73-- 66.91702072108919--- 57.30556133676427--5

2024-09-14 05:40:00 !! 60452.27101716981 !! 60469.65088176663 !! 60409.74
PP 3.9299999999999997-- 21.93706438970611--- 38.144913951611564--60409.74--2024-09-14 05:45:00
PP 6.04-- 19.62727175652647--- 36.465128325283466--60409.74--2024-09-14 05:50:00
PP 7.489999999999999-- 18.09012760615812--- 35.30690256871891--60409.74--2024-09-14 05:55:00
PP -0.4600000000000002-- 45.371220101462875--- 45.49662507947991--60409.74--2024-09-14 06:00:00
PP 4.27-- 36.84748596076718--- 41.32372231439459--60409.74--2024-09-14 06:05:00
PP 0.5199999999999996-- 46.207758678862305--- 45.59041911811754--60409.74--2024-09-14 06:10:00
PP -1.86-- 51.51576533958037--- 48.16058001065291--60409.74--2024-09-14 06:15:00
ema_cross -1.86-- 58.071986136126995--- 48.16058001065291--60409.74 !! 60406.34--2024-09-14 06:15:00
2024-09-14 07:00:00 !! 60363.92137653751 !! 60378.41919779969 !! 60332.33
PP -1.06-- 33.68142199952793--- 40.38881015571018--60332.33--2024-09-14 07:05:00
PP 0.8199999999999998-- 29.67411566670539--- 38.4

2024-09-15 21:15:00 !! 59826.52847824119 !! 59862.208600884805 !! 59792.43
PP -2.12-- 34.46984050616393--- 36.18163714461334--59792.43--2024-09-15 21:20:00
PP -7.19-- 50.25372505308914--- 42.412646187565855--59792.43--2024-09-15 21:25:00
ema_cross -7.19-- 65.31902571979867--- 42.412646187565855--59792.43 !! 59842.34--2024-09-15 21:25:00
2024-09-15 22:30:00 !! 59934.64219277551 !! 59930.42468448443 !! 59868.03
PP -4.890000000000001-- 44.86727763254113--- 47.285962507757525--59868.03--2024-09-15 22:35:00
PP -1.6-- 40.573278821770074--- 45.17401058439234--59868.03--2024-09-15 22:40:00
PP 1.7699999999999996-- 36.393416336230175--- 43.04433870937871--59868.03--2024-09-15 22:45:00
PP -3.74-- 46.82349971481596--- 47.40222091199196--59868.03--2024-09-15 22:50:00
PP -3.5500000000000003-- 46.515089102758225--- 47.267235625865155--59868.03--2024-09-15 22:55:00
PP -2.79-- 45.13324181503008--- 46.6966647794303--59868.03--2024-09-15 23:00:00
PP 4.72-- 33.638486720414235--- 41.391536444194486--59868.

In [361]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

# print(time.time() - t1)
# -6, 6

1470.8749999999998
Total negative sm -->-472.59000000000003
Total negative -->236
Total positive sm -->1943.4650000000004
Total positive -->99
Length 335


In [467]:
profit.sort()

In [498]:
profit

[-1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 8.015,
 6.834999999999999,
 -1,
 -1,
 0.5199999999999996,
 -1,
 6.65,
 -1,
 37.099999999999994,
 16.639999999999997,
 5.43,
 -1,
 -1,
 21.220000000000002,
 -1,
 -1,
 -1,
 -10,
 -1,
 -10,
 -6.94,
 16.035,
 -6.53,
 9.65,
 -1,
 -1,
 -1,
 -1,
 -1,
 -9.17,
 9.915000000000001,
 -1,
 -1,
 -1,
 -1,
 -1,
 41.66,
 -6.4,
 -1,
 -1,
 -1,
 8.31,
 -1,
 -1,
 12.915000000000001,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 18.2,
 -1,
 5.07,
 1.38,
 -1,
 23.3,
 44.199999999999996,
 -1,
 -1,
 8.469999999999999,
 -10,
 -7.75,
 -1,
 -1,
 -10,
 -1,
 -1,
 -1,
 -1,
 5.9799999999999995,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 5.175000000000001,
 10.530000000000001,
 -1,
 -1,
 5.699999999999999,
 -1,
 -1,
 -1,
 -1,
 1.42,
 -1,
 -1,
 -1,
 4.319999999999999,
 -1,
 0.020000000000000018,
 -1,
 -1,
 -1,
 -1,
 -9.34,
 -1,
 7.8999999999999995,
 1.1099999999999999,
 -1,
 -1,
 -1,
 -1,
 16.15,
 -1,
 0.18999999999999995,
 -1,
 -1,
 5.285,
 -1,
 -1,
 -1,
 -1,
 -1,
 -9.629999999999999,
 -1,

In [486]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "USDJPY"
a = get_values(symbol, 20000, 2, 'M30')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0


# 
for i in range(10, len(a)-10):
    if check==0:
        if a.iloc[i].rsi1 <= 50 and a.iloc[i-1].rsi1 <= 50 and a.iloc[i-2].rsi1 <= 50 and a.iloc[i].rsi1 >= 29 and \
            a.iloc[i-1].rsi1 >= 30 and a.iloc[i-2].rsi1 > 30 and a.iloc[i-3].rsi1 >= 50:
            print("SELL")
            if a.iloc[i-4].rsi1 >= 70 :
                if (a.iloc[i-3].open - a.iloc[i-3].close) <0.250 and (a.iloc[i-2].open - a.iloc[i-2].close) <0.250:   
#                 if (a.iloc[i-3].close - a.iloc[i-3].open) <0.250 and (a.iloc[i-2].close - a.iloc[i-2].open) <0.250:             
                    
                    print("+++"*20)

                    print(f"{a.iloc[i].name} -- ")
                    buy_price = a.iloc[i].close
                    check=1
                else:
                    pass
            else:
                print("=="*20)

                print(f"{a.iloc[i].name} -- ")
                buy_price = a.iloc[i].close
                check=1
            
        if a.iloc[i].rsi1 >= 50 and a.iloc[i-1].rsi1 >= 50 and a.iloc[i-2].rsi1 >= 50 and a.iloc[i].rsi1 <= 71 and \
            a.iloc[i-1].rsi1 <= 70 and a.iloc[i-2].rsi1 <70 and a.iloc[i-3].rsi1 <= 50:
            print("BUY")
            if a.iloc[i-4].rsi1 <= 29 :
                if (a.iloc[i-3].close - a.iloc[i-3].open) <0.250 and (a.iloc[i-2].close - a.iloc[i-2].open) <0.250:             

#                 if (a.iloc[i-3].close - a.iloc[i-3].open) <0.250 and (a.iloc[i-2].close - a.iloc[i-2].open) <0.250:             
                    print("+++"*20)

                    print(f"{a.iloc[i].name} -- ")
                    buy_price = a.iloc[i].close
                    check=2
                else:
                    pass
            else:
                print("=="*20)

                print(f"{a.iloc[i].name} -- ")
                buy_price = a.iloc[i].close
                check=2
    elif check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - (2.50*0.1*10)
        print(f"{pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if (buy_price-sell_price) > 0.920:
            pp1 = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - (2.50*0.1*10) 
            print(f"PP {pp1}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp1)
            check = 0
        if (sell_price-buy_price) > 0.100:
            pp1 = price_action(symbol, 0.1, buy_price, buy_price+0.100, mt5.ORDER_TYPE_SELL) -(2.50*0.1*10)
            profit.append(pp1)
            print(f"PP_LOSS {pp1}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            check = 0
        elif a.iloc[i].rsi1 > 85:
            profit.append(pp)
            print(f"RSI {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            check =0
            
        elif a.iloc[i].rsi1 < 10:
            profit.append(pp)
            print(f"RSI_DOne {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            check =0

    elif check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if (sell_price-buy_price) > 0.920:
            pp1 = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - (2.50*0.1*10) 
            print(f"PP {pp1}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp1)
            check = 0
        if (buy_price - sell_price) > 0.100:
            pp1 = price_action(symbol, 0.1, buy_price, buy_price-0.100, mt5.ORDER_TYPE_BUY) -(2.50*0.1*10)
            profit.append(pp)
            print(f"PP_LOSS {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            check = 0
        elif a.iloc[i].rsi1 < 15:
            profit.append(pp)
            print(f"RSI {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            check =0
            
        elif a.iloc[i].rsi1 > 90:
            profit.append(pp)
            print(f"RSI_DOne {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            check =0

BUY
2023-02-15 06:00:00 -- 
3.74--133.0475---133.089--133.006--2023-02-15 06:30:00
3.59--133.088---133.087--133.006--2023-02-15 07:00:00
3.51--133.0865---133.086--133.006--2023-02-15 07:30:00
2.6900000000000004--133.0805---133.075--133.006--2023-02-15 08:00:00
20.45--133.1935---133.312--133.006--2023-02-15 08:30:00
20.3--133.311---133.31--133.006--2023-02-15 09:00:00
22.32--133.3235---133.337--133.006--2023-02-15 09:30:00
20.23--133.32299999999998---133.309--133.006--2023-02-15 10:00:00
28.9--133.36700000000002---133.425--133.006--2023-02-15 10:30:00
24.57--133.39600000000002---133.367--133.006--2023-02-15 11:00:00
23.9--133.3625---133.358--133.006--2023-02-15 11:30:00
21.43--133.3415---133.325--133.006--2023-02-15 12:00:00
25.09--133.34949999999998---133.374--133.006--2023-02-15 12:30:00
23.6--133.364---133.354--133.006--2023-02-15 13:00:00
29.729999999999997--133.395---133.436--133.006--2023-02-15 13:30:00
32.56--133.45499999999998---133.474--133.006--2023-02-15 14:00:00
47.03--133.5

14.77--134.85950000000003---134.889--134.656--2023-02-24 10:00:00
27.49--134.97500000000002---135.061--134.656--2023-02-24 10:30:00
34.79--135.1105---135.16--134.656--2023-02-24 11:00:00
33.09--135.1485---135.137--134.656--2023-02-24 11:30:00
29.630000000000003--135.1135---135.09--134.656--2023-02-24 12:00:00
35.31--135.1285---135.167--134.656--2023-02-24 12:30:00
43.04--135.21949999999998---135.272--134.656--2023-02-24 13:00:00
51.93--135.33249999999998---135.393--134.656--2023-02-24 13:30:00
64.85--135.481---135.569--134.656--2023-02-24 14:00:00
71.0--135.611---135.653--134.656--2023-02-24 14:30:00
PP 71.0--89.44713892515223---135.653--134.656--2023-02-24 14:30:00
SELL
2023-02-27 03:00:00 -- 
2.7199999999999998--34.11833708598594---136.127--136.198--2023-02-27 03:30:00
6.98--30.250794637033366---136.069--136.198--2023-02-27 04:00:00
-1.4--44.640792912347266---136.183--136.198--2023-02-27 04:30:00
-5.22--50.117448376181365---136.235--136.198--2023-02-27 05:00:00
-13.43--60.04895564079

BUY
2023-03-08 19:30:00 -- 
-2.79--137.191---137.189--137.193--2023-03-08 20:00:00
0.4900000000000002--137.2115---137.234--137.193--2023-03-08 20:30:00
2.3100000000000005--137.2465---137.259--137.193--2023-03-08 21:00:00
-0.97--137.23649999999998---137.214--137.193--2023-03-08 21:30:00
3.1100000000000003--137.24200000000002---137.27--137.193--2023-03-08 22:00:00
0.7800000000000002--137.25400000000002---137.238--137.193--2023-03-08 22:30:00
7.84--137.2865---137.335--137.193--2023-03-08 23:00:00
9.08--137.3435---137.352--137.193--2023-03-08 23:30:00
0.56--137.2935---137.235--137.193--2023-03-09 00:00:00
1.7300000000000004--137.243---137.251--137.193--2023-03-09 00:30:00
2.38--137.25549999999998---137.26--137.193--2023-03-09 01:00:00
-0.6099999999999999--137.2395---137.219--137.193--2023-03-09 01:30:00
-21.63--137.075---136.931--137.193--2023-03-09 02:00:00
PP_LOSS -21.63--25.243338201988436---136.931--137.193--2023-03-09 02:00:00
BUY
2023-03-09 18:30:00 -- 
-13.14--136.3295---136.257--13

BUY
2023-03-20 15:30:00 -- 
8.96--131.6505---131.726--131.575--2023-03-20 16:00:00
10.25--131.7345---131.743--131.575--2023-03-20 16:30:00
14.190000000000001--131.769---131.795--131.575--2023-03-20 17:00:00
-1.3599999999999999--131.6925---131.59--131.575--2023-03-20 17:30:00
-10.64--131.529---131.468--131.575--2023-03-20 18:00:00
PP_LOSS -10.64--48.37910894754694---131.468--131.575--2023-03-20 18:00:00
BUY
2023-03-20 19:30:00 -- 
-13.83--131.5505---131.476--131.625--2023-03-20 20:00:00
PP_LOSS -13.83--45.837252753398545---131.476--131.625--2023-03-20 20:00:00
SELL
2023-03-20 21:00:00 -- 
-3.7199999999999998--45.46512948780393---131.419--131.403--2023-03-20 21:30:00
-2.65--44.53614944543648---131.405--131.403--2023-03-20 22:00:00
4.35--38.504380783881615---131.313--131.403--2023-03-20 22:30:00
7.25--36.14538367378266---131.275--131.403--2023-03-20 23:00:00
6.34--37.55486813178603---131.287--131.403--2023-03-20 23:30:00
-2.58--50.087187166175184---131.404--131.403--2023-03-21 00:00:00
-0

BUY
2023-03-30 15:30:00 -- 
-1.6--132.768---132.774--132.762--2023-03-30 16:00:00
-9.51--132.7215---132.669--132.762--2023-03-30 16:30:00
3.45--132.755---132.841--132.762--2023-03-30 17:00:00
-7.62--132.76749999999998---132.694--132.762--2023-03-30 17:30:00
-6.64--132.70049999999998---132.707--132.762--2023-03-30 18:00:00
-6.57--132.70749999999998---132.708--132.762--2023-03-30 18:30:00
-24.32--132.59050000000002---132.473--132.762--2023-03-30 19:00:00
PP_LOSS -24.32--36.057382834935225---132.473--132.762--2023-03-30 19:00:00
SELL
2023-03-30 20:00:00 -- 
-2.42--36.43266739664271---132.429--132.43--2023-03-30 20:30:00
-1.8199999999999998--35.84146598089421---132.421--132.43--2023-03-30 21:00:00
-3.26--38.60209745521882---132.44--132.43--2023-03-30 21:30:00
-8.61--48.30034350160537---132.511--132.43--2023-03-30 22:00:00
-16.0--58.784139295787135---132.609--132.43--2023-03-30 22:30:00
PP_LOSS -10.05--58.784139295787135---132.609--132.43--2023-03-30 22:30:00
BUY
2023-03-30 23:30:00 -- 
-4.

...

17.02--26.23644530759232---155.775--156.079--2024-07-23 19:30:00
28.48--19.60545095591354---155.597--156.079--2024-07-23 20:00:00
23.07--29.42577573425747---155.681--156.079--2024-07-23 20:30:00
21.46--32.2973016363305---155.706--156.079--2024-07-23 21:00:00
27.19--27.628359322367416---155.617--156.079--2024-07-23 21:30:00
27.25--27.5761026274717---155.616--156.079--2024-07-23 22:00:00
26.41--29.595751410862064---155.629--156.079--2024-07-23 22:30:00
28.35--27.528907725044846---155.599--156.079--2024-07-23 23:00:00
29.38--26.382498426197344---155.583--156.079--2024-07-23 23:30:00
30.799999999999997--24.73041629168459---155.561--156.079--2024-07-24 00:00:00
30.28--26.67829698608385---155.569--156.079--2024-07-24 00:30:00
27.96--35.44847927960954---155.605--156.079--2024-07-24 01:00:00
26.41--40.942689501764924---155.629--156.079--2024-07-24 01:30:00
20.17--57.85636707627076---155.726--156.079--2024-07-24 02:00:00
18.43--61.442391862956576---155.753--156.079--2024-07-24 02:30:00
11.42--7

BUY
2024-08-05 23:00:00 -- 
6.800000000000001--144.09300000000002---144.16--144.026--2024-08-05 23:30:00
5.550000000000001--144.151---144.142--144.026--2024-08-06 00:00:00
0.3500000000000001--144.1045---144.067--144.026--2024-08-06 00:30:00
-12.86--143.972---143.877--144.026--2024-08-06 01:00:00
PP_LOSS -12.86--50.93656592411591---143.877--144.026--2024-08-06 01:00:00
SELL
2024-08-06 11:30:00 -- 
1.0899999999999999--36.05842413428836---144.861--144.913--2024-08-06 12:00:00
9.31--33.1050504795535---144.742--144.913--2024-08-06 12:30:00
1.0899999999999999--38.93973192540518---144.861--144.913--2024-08-06 13:00:00
-0.77--40.317680795905396---144.888--144.913--2024-08-06 13:30:00
-35.24--59.905314049141396---145.389--144.913--2024-08-06 14:00:00
PP_LOSS -9.4--59.905314049141396---145.389--144.913--2024-08-06 14:00:00
BUY
2024-08-06 19:30:00 -- 
-16.48--145.2615---145.16--145.363--2024-08-06 20:00:00
PP_LOSS -16.48--57.79676436687521---145.16--145.363--2024-08-06 20:00:00
BUY
2024-08-07 11:

SELL
2024-08-16 18:30:00 -- 
0.94--45.33622774283445---148.073--148.124--2024-08-16 19:00:00
6.42--41.41458048830376---147.992--148.124--2024-08-16 19:30:00
22.27--32.06600182284812---147.758--148.124--2024-08-16 20:00:00
23.9--31.22265857097254---147.734--148.124--2024-08-16 20:30:00
22.88--32.516800865983626---147.749--148.124--2024-08-16 21:00:00
28.92--28.769531035300872---147.66--148.124--2024-08-16 21:30:00
27.5--30.959740483513258---147.681--148.124--2024-08-16 22:00:00
26.82--32.119301593632514---147.691--148.124--2024-08-16 22:30:00
32.8--27.395423268571463---147.603--148.124--2024-08-16 23:00:00
32.59--27.817653163918166---147.606--148.124--2024-08-16 23:30:00
28.52--36.44210099592397---147.666--148.124--2024-08-19 00:00:00
28.92--35.941100151516466---147.66--148.124--2024-08-19 00:30:00
13.19--60.46189085825435---147.892--148.124--2024-08-19 01:00:00
15.829999999999998--56.23984883948288---147.853--148.124--2024-08-19 01:30:00
8.04--64.71599951876014---147.968--148.124--2024

11.57--144.37---144.304--144.101--2024-08-29 01:00:00
10.46--144.296---144.288--144.101--2024-08-29 01:30:00
12.26--144.301---144.314--144.101--2024-08-29 02:00:00
20.07--144.3705---144.427--144.101--2024-08-29 02:30:00
21.18--144.435---144.443--144.101--2024-08-29 03:00:00
19.45--144.4305---144.418--144.101--2024-08-29 03:30:00
22.9--144.44299999999998---144.468--144.101--2024-08-29 04:00:00
33.46--144.5445---144.621--144.101--2024-08-29 04:30:00
26.15--144.56799999999998---144.515--144.101--2024-08-29 05:00:00
30.35--144.5455---144.576--144.101--2024-08-29 05:30:00
24.35--144.5325---144.489--144.101--2024-08-29 06:00:00
26.22--144.5025---144.516--144.101--2024-08-29 06:30:00
28.63--144.5335---144.551--144.101--2024-08-29 07:00:00
36.28--144.60649999999998---144.662--144.101--2024-08-29 07:30:00
42.82--144.7095---144.757--144.101--2024-08-29 08:00:00
32.49--144.68200000000002---144.607--144.101--2024-08-29 08:30:00
29.18--144.583---144.559--144.101--2024-08-29 09:00:00
29.869999999999

56.39--30.479001544794485---142.298--143.136--2024-09-10 22:30:00
53.56--35.867392371996516---142.338--143.136--2024-09-10 23:00:00
46.36--47.8844328891532---142.44--143.136--2024-09-10 23:30:00
51.09--41.87183643773732---142.373--143.136--2024-09-11 00:00:00
48.48--46.222375137221675---142.41--143.136--2024-09-11 00:30:00
48.48--46.222375137221675---142.41--143.136--2024-09-11 01:00:00
46.5--50.07144107916397---142.438--143.136--2024-09-11 01:30:00
57.8--33.89713977128099---142.278--143.136--2024-09-11 02:00:00
61.620000000000005--30.07222652408656---142.224--143.136--2024-09-11 02:30:00
73.24--21.483052487927935---142.06--143.136--2024-09-11 03:00:00
PP 73.24--21.483052487927935---142.06--143.136--2024-09-11 03:00:00
BUY
2024-09-11 13:30:00 -- 
5.9--141.62650000000002---141.686--141.567--2024-09-11 14:00:00
5.41--141.6825---141.679--141.567--2024-09-11 14:30:00
11.04--141.719---141.759--141.567--2024-09-11 15:00:00
47.75--142.0205---142.282--141.567--2024-09-11 15:30:00
50.27--142.3-

In [487]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

# print(time.time() - t1)
# -6, 6

2220.36
Total negative sm -->-6500.429999999998
Total negative -->497
Total positive sm -->8720.789999999999
Total positive -->131
Length 628


In [273]:
profit.sort()

In [274]:
profit

[-68.01,
 -52.9,
 -44.79,
 -42.34,
 -27.41,
 -23.89,
 -22.56,
 -21.57,
 -20.57,
 -20.35,
 -19.85,
 -19.66,
 -18.97,
 -18.72,
 -18.54,
 -17.93,
 -17.560000000000002,
 -16.96,
 -16.7,
 -16.560000000000002,
 -16.48,
 -16.42,
 -15.86,
 -15.36,
 -15.22,
 -14.74,
 -14.68,
 -14.36,
 -14.34,
 -14.2,
 -13.46,
 -13.27,
 -13.24,
 -13.13,
 -13.1,
 -13.07,
 -13.02,
 -13.01,
 -12.95,
 -12.86,
 -12.85,
 -12.82,
 -12.71,
 -12.53,
 -12.52,
 -12.49,
 -12.41,
 -12.15,
 -12.0,
 -11.99,
 -11.97,
 -11.82,
 -11.81,
 -11.76,
 -11.73,
 -11.67,
 -11.59,
 -11.33,
 -11.31,
 -11.21,
 -11.13,
 -10.89,
 -10.56,
 -10.54,
 -10.44,
 -10.36,
 -10.18,
 -10.17,
 -10.14,
 -10.0,
 -9.91,
 -9.879999999999999,
 -9.870000000000001,
 -9.85,
 -9.65,
 -9.620000000000001,
 -9.559999999999999,
 -9.46,
 -9.33,
 -9.29,
 -9.219999999999999,
 -9.19,
 -9.04,
 -8.92,
 23.95,
 47.18,
 48.66,
 48.83,
 49.7,
 55.64,
 56.09,
 56.3,
 56.55,
 57.08,
 57.51,
 57.59,
 58.48,
 58.5,
 62.879999999999995,
 63.06,
 64.73,
 64.73,
 66.33,
 69.34,
 70

8250.05
Total negative sm -->-1200.15
Total negative -->801
Total positive sm -->9450.200000000003
Total positive -->234
Length 1035
